In [21]:
import pandas as pd
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, precision_recall_curve
)
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler 


In [2]:
spam_data=pd.read_csv(r"D:\Complete_proj\Email_spam\model\data\spam_data2.csv")

In [3]:
spam_data.shape

(83446, 6)

In [4]:
spam_data.head(3)

,label,text,num_characters,num_words,num_sentences,transformed_text
0,1,ounce feather bowl hummingbird opec moment ala...,148,20,1,ounc feather bowl hummingbird opec moment alab...
1,1,wulvob get your medircations online qnb ikud v...,808,104,1,wulvob get medirc onlin qnb ikud viagra escape...
2,0,computer connection from cnn com wednesday es...,2235,338,1,comput connect cnn com wednesday escapenumb ma...


In [5]:
spam_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 83446 entries, 0 to 83445
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   label             83446 non-null  int64 
 1   text              83446 non-null  object
 2   num_characters    83446 non-null  int64 
 3   num_words         83446 non-null  int64 
 4   num_sentences     83446 non-null  int64 
 5   transformed_text  83427 non-null  object
dtypes: int64(4), object(2)
memory usage: 3.8+ MB


In [6]:
spam_data.isnull().sum()

label                0
text                 0
num_characters       0
num_words            0
num_sentences        0
transformed_text    19
dtype: int64

In [7]:
spam_data.dropna(subset=['transformed_text'], inplace=True)
print(spam_data.isnull().sum())

label               0
text                0
num_characters      0
num_words           0
num_sentences       0
transformed_text    0
dtype: int64


In [8]:
spam_data.duplicated(subset=['transformed_text']).sum()

np.int64(551)

In [9]:
duplicate_rows=spam_data[spam_data.duplicated(subset=['transformed_text'], keep=False)]
print(duplicate_rows.sort_values(by='transformed_text').head(10))

       label                                               text  \
19198      1  1 - 888 - 662 - 2256\ngtts , supplier of\nlase...   
2363       1  1 - 888 - 662 - 2256\ngtts , supplier of\nlase...   
30906      0  2001\njulie :\ni talked to both maureen and ta...   
7397       0  2001\njulie :\ni talked to both maureen and ta...   
18025      0  as of the 4 th april 2001 we will be changing ...   
43977      0  as of the 4 th april 2001 we will be changing ...   
5384       0  9868\nplease note the following for april prod...   
15219      0  9868\nplease note the following for april prod...   
57677      0  we agree with 35 . 000 for april 1 , 2001\n" e...   
7345       0  we agree with 35 . 000 for april 1 , 2001\n" e...   

       num_characters  num_words  num_sentences  \
19198            1975        570             27   
2363             1975        570             27   
30906            4425        913             44   
7397             4425        913             44   
18025  

In [10]:
spam_data=spam_data.drop_duplicates(subset=['transformed_text'], keep='first')

In [11]:
spam_data.duplicated(subset=['transformed_text']).sum()

np.int64(0)

In [12]:
spam_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 82876 entries, 0 to 83445
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   label             82876 non-null  int64 
 1   text              82876 non-null  object
 2   num_characters    82876 non-null  int64 
 3   num_words         82876 non-null  int64 
 4   num_sentences     82876 non-null  int64 
 5   transformed_text  82876 non-null  object
dtypes: int64(4), object(2)
memory usage: 4.4+ MB


In [13]:
spam_data.label.value_counts()

label
1    43726
0    39150
Name: count, dtype: int64

In [14]:
spam_data.head(3)

,label,text,num_characters,num_words,num_sentences,transformed_text
0,1,ounce feather bowl hummingbird opec moment ala...,148,20,1,ounc feather bowl hummingbird opec moment alab...
1,1,wulvob get your medircations online qnb ikud v...,808,104,1,wulvob get medirc onlin qnb ikud viagra escape...
2,0,computer connection from cnn com wednesday es...,2235,338,1,comput connect cnn com wednesday escapenumb ma...


### Data is processed and safe to split into training and test sets

In [17]:
x_train, x_test, y_train, y_test=train_test_split(spam_data['transformed_text'], spam_data['label'], test_size=0.2, random_state=42)

In [18]:
x_train

7        hi error tr sampl escapenumb escapenumb escape...
65676    new ticket creat jame keenan pleas includ stri...
63714    cdpn move corner gp market china china datacom...
79926    taken oar tri revis list pleas look make chang...
70245    wither good intent like violent escapenumb uns...
                               ...                        
6272     jeff got 20 cent swith option per dth assumpt ...
55132    dear hescapenumberm escapenumberwn tire pay hi...
77306    advertis brought experi sunset resort disconti...
860      altern medicin databas escapenumb million reco...
15821    alert lookup cdyv current escapenumb escapenum...
Name: transformed_text, Length: 66300, dtype: object

In [19]:
y_train

7        0
65676    0
63714    1
79926    0
70245    1
        ..
6272     0
55132    1
77306    1
860      0
15821    1
Name: label, Length: 66300, dtype: int64

In [25]:
param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__max_features": [1000, 3000, 5000],
    

    "clf__loss": ["log_loss","hinge"],
    "clf__penalty": ["l2","elasticnet"],
    "clf__alpha": [1e-4, 1e-3, 1e-2],
    "clf__max_iter": [1000,2000]
}
# "tfidf__stop_words": ["english", None],
    # "tfidf__max_df": [0.7, 0.8, 0.9],

In [26]:
pipeline=Pipeline([
    ('tfidf',TfidfVectorizer()), 
    ('scaler', StandardScaler(with_mean=False)),
    ('clf', SGDClassifier(random_state=42))
])

In [27]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import classification_report, confusion_matrix

# 1. Initialize the RandomizedSearch
# n_iter=50 means it will only do 250 fits (50 combos * 5 folds) instead of 4320
random_search = RandomizedSearchCV(
    estimator=pipeline, 
    param_distributions=param_grid, 
    n_iter=50, 
    scoring='f1_weighted', # Better for spam (imbalanced data) than simple accuracy
    cv=5, 
    verbose=1, 
    n_jobs=-1, 
    random_state=42
)

# 2. Fit on your training data
print("Starting optimized search...")
random_search.fit(x_train, y_train)

# 3. Get the results
print(f"Best Score: {random_search.best_score_}")
print(f"Best Params: {random_search.best_params_}")

# 4. Evaluate on Test Data
best_model = random_search.best_estimator_
y_pred = best_model.predict(x_test)

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred))

Starting optimized search...
Fitting 5 folds for each of 50 candidates, totalling 250 fits


KeyboardInterrupt: 

In [23]:
grid=GridSearchCV(
    pipeline,
    param_grid, 
    scoring="f1_macro", 
    cv=5, 
    n_jobs=-1, 
    verbose=1
)

In [24]:
grid.fit(x_train, y_train)
print(f"\n  Best params : {grid.best_params_}")
print(f"  Best CV F1  : {grid.best_score_:.4f}")

Fitting 5 folds for each of 864 candidates, totalling 4320 fits


KeyboardInterrupt: 